# This notebook fit a GEV to maximum annual percipitation using MOM
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [1]:
import numpy as np
import pandas as pd
import lmoments3 as lm
from lmoments3 import distr
import os
from scipy.stats import genextreme

# Read data

## AORC

In [3]:
# --- Paths ---
BASE_DIR = r"D:\BMM-IDF4Drainage_data_results\Data\AORC"
CITIES_CSV = r"D:\BMM-IDF4Drainage_data_results\us_cities_coordinates.csv"

# --- Load city list ---
cities_df = pd.read_csv(CITIES_CSV)
city_names = cities_df["City"].astype(str).str.strip().tolist()

# --- Final container ---
dataframes = {}

# --- Helper to read a CSV safely ---
def read_if_exists(path, usecols=None):
    if os.path.exists(path):
        return pd.read_csv(path, usecols=usecols)
    return None

# --- Build combined structure ---
for city_full in city_names:
    # Strip the state to get short name
    city_short = city_full.split(",")[0].strip()

    # Paths to the hourly and daily data
    hourly_path = os.path.join(BASE_DIR, city_full, f"{city_full}_annual_max_hourly.csv")
    daily_path  = os.path.join(BASE_DIR, city_full, f"{city_full}_annual_max_daily.csv")

    df_hourly = read_if_exists(hourly_path, usecols=["year", "max_hourly_mm"])
    df_daily  = read_if_exists(daily_path,  usecols=["year", "max_daily_mm"])

    if df_hourly is not None and df_daily is not None:
        # Keep only the common years between both
        df = pd.merge(df_hourly, df_daily, on="year", how="inner")

        # Rename columns to match your original structure
        df.rename(columns={"max_hourly_mm": "1h", "max_daily_mm": "24h"}, inplace=True)

        # Filter to 1979–2024 (if desired)
        df = df[df["year"].between(1979, 2024)]

        # Sort by year and reset index
        df = df.sort_values("year").reset_index(drop=True)

        # Save to dict with short name
        dataframes[city_short] = df

# --- print ---
print(dataframes["Birmingham"].head())
print(dataframes["Phoenix"])
print(list(dataframes.keys()))
# cities names
cities = ['Birmingham', 'Phoenix', 'Little Rock', 'Los Angeles', 'Denver', 'Hartford', 'Wilmington', 'Miami', 'Atlanta', 'Boise',
           'Chicago', 'Indianapolis', 'Des Moines', 'Wichita', 'Louisville', 'New Orleans', 'Portland', 'Baltimore', 'Boston', 'Detroit',
             'Minneapolis', 'Jackson', 'St. Louis', 'Billings', 'Omaha', 'Las Vegas', 'Manchester', 'Newark', 'Albuquerque', 'New York City',
               'Charlotte', 'Fargo', 'Columbus', 'Oklahoma City', 'Philadelphia', 'Providence', 'Charleston', 'Sioux Falls', 'Nashville', 'Houston',
                 'Salt Lake City', 'Burlington', 'Richmond', 'Seattle', 'Milwaukee', 'Cheyenne']             

   year         1h         24h
0  1979  21.200000  135.500002
1  1980  18.300000  102.500002
2  1981  43.200001   77.800001
3  1982  35.500001   74.100001
4  1983  24.100000  103.300002
    year         1h        24h
0   1979   8.400000  31.300000
1   1980   3.500000  14.100000
2   1981   4.600000  22.600000
3   1982   6.800000  24.700000
4   1983   9.700000  40.700001
5   1984   8.400000  32.300000
6   1985  10.400000  27.200000
7   1986   5.700000  28.900000
8   1987  15.000000  34.300001
9   1988   4.700000  26.100000
10  1989  20.400000  29.400000
11  1990   7.900000  29.400000
12  1991  20.200000  25.200000
13  1992  13.500000  42.400001
14  1993   7.700000  35.000001
15  1994  10.800000  15.300000
16  1995   6.800000  21.200000
17  1996   5.900000  11.300000
18  1997  11.300000  17.600000
19  1998   9.500000  18.200000
20  1999  11.900000  26.600000
21  2000   8.300000  51.100001
22  2001   6.000000  16.800000
23  2002   7.200000  13.700000
24  2003  16.100000  32.700000
25  2004

# CMIP6

In [2]:
# --- Paths ---
BASE_DIR = r"D:\BMM-IDF4Drainage_data_results\Data\CMIP6"
CITIES_CSV = r"D:\BMM-IDF4Drainage_data_results\us_cities_coordinates.csv"

# --- Load city list ---
cities_df = pd.read_csv(CITIES_CSV)
city_names = cities_df["City"].astype(str).str.strip().tolist()

# --- Final container ---
dataframes_ssp = {}

# --- Helper to read a CSV safely ---
def read_if_exists(path, usecols=None):
    if os.path.exists(path):
        return pd.read_csv(path, usecols=usecols)
    return None

# --- Build combined structure ---
for city_full in city_names:
    # Strip the state to get short name
    city_short = city_full.split(",")[0].strip()

    # Paths to the hourly and daily data
    daily_path_126  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP126_yearly_max.csv")
    daily_path_370  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP370_yearly_max.csv")
    daily_path_585  = os.path.join(BASE_DIR, city_full, f"{city_full}_SSP585_yearly_max.csv")


    df_daily_126  = read_if_exists(daily_path_126,  usecols=["year", "max_daily_mm"])
    df_daily_370  = read_if_exists(daily_path_370,  usecols=["year", "max_daily_mm"])
    df_daily_585  = read_if_exists(daily_path_585,  usecols=["year", "max_daily_mm"])

    if (df_daily_126 is not None) and (df_daily_370 is not None) and (df_daily_585 is not None):

        # rename columns so they don't collide, then merge pairwise
        df_daily_126 = df_daily_126.rename(columns={"max_daily_mm": "126"})
        df_daily_370 = df_daily_370.rename(columns={"max_daily_mm": "370"})
        df_daily_585 = df_daily_585.rename(columns={"max_daily_mm": "585"})

        # Merge two at a time
        df = df_daily_126.merge(df_daily_370, on="year", how="inner").merge(df_daily_585, on="year", how="inner")

        # Optional filter range (CMIP6 futures typically 2015–2100)
        df = df[df["year"].between(2015, 2100)]

        df = df.sort_values("year").reset_index(drop=True)
        # --- Split each scenario into two new columns ---
        df["126_2025_2100"] = np.where(df["year"].between(2025, 2100), df["126"], np.nan)
        df["126_2025_2064"] = np.where(df["year"].between(2025, 2064), df["126"], np.nan)
        df["126_2065_2100"] = np.where(df["year"].between(2065, 2100), df["126"], np.nan)

        df["370_2025_2100"] = np.where(df["year"].between(2025, 2100), df["370"], np.nan)
        df["370_2025_2064"] = np.where(df["year"].between(2025, 2064), df["370"], np.nan)
        df["370_2065_2100"] = np.where(df["year"].between(2065, 2100), df["370"], np.nan)

        df["585_2025_2100"] = np.where(df["year"].between(2025, 2100), df["585"], np.nan)
        df["585_2025_2064"] = np.where(df["year"].between(2025, 2064), df["585"], np.nan)
        df["585_2065_2100"] = np.where(df["year"].between(2065, 2100), df["585"], np.nan)
        df["585_2075_2100"] = np.where(df["year"].between(2075, 2100), df["585"], np.nan)

        dataframes_ssp[city_short] = df

# --- print ---
print(dataframes_ssp["Birmingham"].head())
print(dataframes_ssp["Phoenix"])
print(list(dataframes_ssp.keys()))
# cities names
cities = ['Birmingham', 'Phoenix', 'Little Rock', 'Los Angeles', 'Denver', 'Hartford', 'Wilmington', 'Miami', 'Atlanta', 'Boise',
           'Chicago', 'Indianapolis', 'Des Moines', 'Wichita', 'Louisville', 'New Orleans', 'Portland', 'Baltimore', 'Boston', 'Detroit',
             'Minneapolis', 'Jackson', 'St. Louis', 'Billings', 'Omaha', 'Las Vegas', 'Manchester', 'Newark', 'Albuquerque', 'New York City',
               'Charlotte', 'Fargo', 'Columbus', 'Oklahoma City', 'Philadelphia', 'Providence', 'Charleston', 'Sioux Falls', 'Nashville', 'Houston',
                 'Salt Lake City', 'Burlington', 'Richmond', 'Seattle', 'Milwaukee', 'Cheyenne']             



   year        126         370        585  126_2025_2100  126_2025_2064  \
0  2015  62.940857  114.352142  92.040123            NaN            NaN   
1  2016  81.798233   61.884373  62.521687            NaN            NaN   
2  2017  44.011440   66.001343  58.154327            NaN            NaN   
3  2018  45.637985   64.430138  86.788147            NaN            NaN   
4  2019  77.624031   63.680565  60.768650            NaN            NaN   

   126_2065_2100  370_2025_2100  370_2025_2064  370_2065_2100  585_2025_2100  \
0            NaN            NaN            NaN            NaN            NaN   
1            NaN            NaN            NaN            NaN            NaN   
2            NaN            NaN            NaN            NaN            NaN   
3            NaN            NaN            NaN            NaN            NaN   
4            NaN            NaN            NaN            NaN            NaN   

   585_2025_2064  585_2065_2100  585_2075_2100  
0            NaN   

# Parameter Inference under  MOM

## AORC

### 1hr

In [26]:
#hourly return levels and parameters
T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T
rows = []

for i in (range(len(cities))):
    city=cities[i]
    data1 = np.array(dataframes.get(cities[i])["1h"])
    params = distr.gev.lmom_fit(data1)
    q = genextreme.ppf(p, params["c"], loc=params["loc"], scale=params["scale"])
    rows.append([city, *q, params["c"], params["loc"], params["scale"]])
df = pd.DataFrame(rows, columns=["City"] + [str(int(t)) for t in T]+ ["shape(c)", "loc", "scale"])
df.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_1h_AORC_MOM.csv', index=False)
df.head(50)


,City,2,5,10,25,50,100,shape(c),loc,scale
0,Birmingham,26.339315,37.707019,44.661864,52.847665,58.514483,63.819265,0.083626,22.426759,10.839513
1,Phoenix,9.589037,14.926267,19.361799,26.257585,32.504867,39.858193,-0.243199,8.154969,3.740943
2,Little Rock,25.574017,36.891292,44.580991,54.526906,62.074926,69.713171,-0.027531,21.989301,9.731336
3,Los Angeles,13.940675,20.232863,24.215533,29.048018,32.494432,35.802928,0.047700,11.832114,5.803469
4,Denver,9.702850,16.738610,22.319767,30.632504,37.860825,46.075295,-0.193082,7.737939,5.173638
5,Hartford,17.077112,26.515287,34.068511,45.410086,55.349046,66.719277,-0.202566,14.460420,6.877677
6,Wilmington,21.857093,31.622331,37.933570,45.736361,51.403740,56.929221,0.025606,18.638211,8.823728
7,Miami,24.058414,39.220549,51.433597,69.881991,86.141586,104.833912,-0.209550,19.877317,10.975308
8,Atlanta,19.947792,30.010804,37.437480,47.791858,56.241477,65.337008,-0.115691,16.966094,7.964056
9,Boise,6.505715,9.446497,11.642938,14.738819,17.291875,20.064829,-0.128478,5.642799,2.299398


### 24hr

In [27]:
#Daily return levels and parameters
T = np.array([2, 5, 10, 25, 50, 100])
p = 1 - 1 / T
rows = []

for i in (range(len(cities))):
    city=cities[i]
    data1 = np.array(dataframes.get(cities[i])["24h"])
    params = distr.gev.lmom_fit(data1)
    q = genextreme.ppf(p, params["c"], loc=params["loc"], scale=params["scale"])
    rows.append([city, *q, params["c"], params["loc"], params["scale"]])
df = pd.DataFrame(rows, columns=["City"] + [str(int(t)) for t in T]+ ["shape(c)", "loc", "scale"])
df.to_csv(r'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_24h_AORC_MOM.csv', index=False)
df.head(50)

,City,2,5,10,25,50,100,shape(c),loc,scale
0,Birmingham,79.788003,108.107739,127.031638,151.141842,169.173428,187.195179,-0.009797,70.697414,24.758412
1,Phoenix,24.709652,36.620753,45.766343,58.986932,70.157755,82.545131,-0.158091,21.292723,9.055323
2,Little Rock,78.611683,106.512937,127.599320,157.642498,182.676272,210.107923,-0.141107,70.503092,21.556457
3,Los Angeles,58.430395,88.350546,107.941608,132.448588,150.453218,168.177965,0.011780,48.669411,26.689568
4,Denver,34.472200,45.719392,53.172374,62.596490,69.593074,76.542411,-0.000907,30.837702,9.914779
5,Hartford,66.465041,84.612641,96.959239,112.947077,125.094103,137.398162,-0.028904,60.722777,15.584448
6,Wilmington,64.176889,86.737314,103.835500,128.259295,148.661194,171.064670,-0.144128,57.635541,17.380277
7,Miami,74.979484,112.463195,149.995635,219.417704,293.571290,394.141843,-0.445475,66.382971,21.592184
8,Atlanta,68.695312,89.919415,105.185780,125.978081,142.566631,160.082930,-0.088231,62.273974,17.238336
9,Boise,21.474189,27.961071,32.373679,38.086827,42.426884,46.822528,-0.028736,19.421350,5.571559


## CMIP6

### 24hr

In [4]:
for scen in ['126_2025_2064','126_2065_2100','370_2025_2064','370_2065_2100','585_2025_2064','585_2065_2100','585_2025_2100']:
    #hourly return levels and parameters
    T = np.array([2, 5, 10, 25, 50, 100])
    p = 1 - 1 / T
    rows = []
    for i in (range(len(cities))):
        city=cities[i]
        data24h = np.array(dataframes_ssp.get(cities[i])[scen])
        data1 = data24h[~np.isnan(data24h)]
        params = distr.gev.lmom_fit(data1)
        q = genextreme.ppf(p, params["c"], loc=params["loc"], scale=params["scale"])
        rows.append([city, *q, params["c"], params["loc"], params["scale"]])
    df = pd.DataFrame(rows, columns=["City"] + [str(int(t)) for t in T]+ ["shape(c)", "loc", "scale"])
    df.to_csv(fr'D:\BMM-IDF4Drainage_data_results\Percentile\return_levels(mm)_1h_CMIP6_{scen}_MOM.csv', index=False)